# Flask: Side-by-Side Comparison

This notebook covers:

1. Implementing the **same three endpoints** in both Flask and FastAPI, exercised in-process
2. Validation: manual `request.get_json()` checks vs Pydantic at the parameter signature
3. Async support: where Flask is now, what `async def` views actually buy you, and where FastAPI is structurally different
4. OpenAPI: nothing for free in Flask, fully automatic in FastAPI — and what that gap means in practice
5. When Flask is *still* the right call
6. A concrete migration recipe for moving a small Flask app to FastAPI without a rewrite

**Scope**: Flask 3 alongside FastAPI, both via their built-in test clients (`app.test_client()` for Flask, `TestClient` for FastAPI). No servers; everything runs in-process so the comparison is apples-to-apples.

## 1. The Same Three Endpoints, Both Frameworks

The fairest comparison is to build the **same feature** in both, with the same domain model. We'll implement three endpoints from the asset/portfolio domain used throughout this curriculum:

- `GET /assets` — list all assets
- `GET /assets/<ticker>` — fetch one
- `POST /assets` — create one, with validation

Both versions live in the same notebook cell so you can read the line-for-line difference. Read it twice: once for the boilerplate, once for the *defaults*. Flask defaults to "you do it"; FastAPI defaults to "I do it".

In [1]:
from flask import Flask, jsonify, request, abort
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

# ----- shared in-memory store --------------------------------------------------
SEED = [
    {"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0},
    {"ticker": "MSFT", "name": "Microsoft",  "price": 420.0},
]

# ----- FLASK version -----------------------------------------------------------
flask_app = Flask(__name__)
flask_store: dict[str, dict] = {a["ticker"]: dict(a) for a in SEED}

@flask_app.get("/assets")
def f_list_assets():
    return jsonify(list(flask_store.values()))

@flask_app.get("/assets/<ticker>")
def f_get_asset(ticker: str):
    asset = flask_store.get(ticker.upper())
    if asset is None:
        # abort() raises an HTTPException — same idea as FastAPI's HTTPException.
        abort(404, description=f"Asset {ticker!r} not found")
    return jsonify(asset)

@flask_app.post("/assets")
def f_create_asset():
    # Flask gives you raw bytes — JSON parsing + every field validation is yours.
    if not request.is_json:
        abort(415, description="Content-Type must be application/json")
    payload = request.get_json(silent=True)
    if not isinstance(payload, dict):
        abort(400, description="JSON object required")
    ticker = payload.get("ticker")
    name   = payload.get("name")
    price  = payload.get("price")
    if not isinstance(ticker, str) or not ticker.isupper() or not (1 <= len(ticker) <= 10):
        abort(422, description="ticker must be 1-10 uppercase letters")
    if not isinstance(name, str) or not name:
        abort(422, description="name must be a non-empty string")
    if not isinstance(price, (int, float)) or price < 0:
        abort(422, description="price must be a non-negative number")
    if ticker in flask_store:
        abort(409, description=f"{ticker} already exists")
    asset = {"ticker": ticker, "name": name, "price": float(price)}
    flask_store[ticker] = asset
    return jsonify(asset), 201

# ----- FASTAPI version --------------------------------------------------------
class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

fa_app = FastAPI()
fa_store: dict[str, Asset] = {a["ticker"]: Asset(**a) for a in SEED}

@fa_app.get("/assets", response_model=list[Asset])
def fa_list_assets():
    return list(fa_store.values())

@fa_app.get("/assets/{ticker}", response_model=Asset)
def fa_get_asset(ticker: str):
    asset = fa_store.get(ticker.upper())
    if asset is None:
        raise HTTPException(404, detail=f"Asset {ticker!r} not found")
    return asset

@fa_app.post("/assets", response_model=Asset, status_code=201)
def fa_create_asset(asset: Asset):
    # Pydantic already validated `asset` before this body runs. If anything
    # was malformed the client got a structured 422 and we never got here.
    if asset.ticker in fa_store:
        raise HTTPException(409, detail=f"{asset.ticker} already exists")
    fa_store[asset.ticker] = asset
    return asset

# ----- exercise both via their respective test clients ------------------------
fc = flask_app.test_client()    # Werkzeug test client (WSGI)
ac = TestClient(fa_app)         # Starlette test client (ASGI)

def show(label, status, body):
    print(f"{label:<20} -> {status}  {body}")

print("HAPPY PATH")
show("flask GET list",  fc.get("/assets").status_code,        fc.get("/assets").get_json())
show("fast  GET list",  ac.get("/assets").status_code,        ac.get("/assets").json())

show("flask GET AAPL",  fc.get("/assets/AAPL").status_code,   fc.get("/assets/AAPL").get_json())
show("fast  GET AAPL",  ac.get("/assets/AAPL").status_code,   ac.get("/assets/AAPL").json())

print("\nVALIDATION FAILURE (price negative)")
bad = {"ticker": "TSLA", "name": "Tesla", "price": -1}
show("flask POST bad",  fc.post("/assets", json=bad).status_code,  fc.post("/assets", json=bad).get_json())
show("fast  POST bad",  ac.post("/assets", json=bad).status_code,  ac.post("/assets", json=bad).json())


HAPPY PATH
flask GET list       -> 200  [{'name': 'Apple Inc.', 'price': 190.0, 'ticker': 'AAPL'}, {'name': 'Microsoft', 'price': 420.0, 'ticker': 'MSFT'}]
fast  GET list       -> 200  [{'ticker': 'AAPL', 'name': 'Apple Inc.', 'price': 190.0}, {'ticker': 'MSFT', 'name': 'Microsoft', 'price': 420.0}]
flask GET AAPL       -> 200  {'name': 'Apple Inc.', 'price': 190.0, 'ticker': 'AAPL'}
fast  GET AAPL       -> 200  {'ticker': 'AAPL', 'name': 'Apple Inc.', 'price': 190.0}

VALIDATION FAILURE (price negative)
flask POST bad       -> 422  None
fast  POST bad       -> 422  {'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'price'], 'msg': 'Input should be greater than or equal to 0', 'input': -1, 'ctx': {'ge': 0.0}}]}


C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Read the two `create_asset` bodies side by side: about 15 lines of validation in Flask, three meaningful lines in FastAPI plus a 4-line Pydantic model. Both correctly reject the negative price; only FastAPI gives the client a **structured error** that says *which field* and *why* (Pydantic's `loc`/`msg`/`type` shape, see notebook 1.3).

That structured error is the most visible artifact of the deeper structural difference: FastAPI is built around **type-driven dispatch** (path/body/query parameters are typed function parameters; the framework parses, validates, and documents them in one pass), while Flask is built around **request-object inspection** (you write code that reads from a global `request`). Both work; one scales better as your endpoint count grows.

## 2. Validation: Manual vs Pydantic

Let's zoom in on the validation cost. Both frameworks accept a JSON body; the difference is who's responsible for proving it conforms.

In Flask, you do:

```python
payload = request.get_json(silent=True)
if not isinstance(payload, dict):           abort(400, ...)
if not isinstance(payload.get("price"), (int, float)) or payload["price"] < 0:
    abort(422, ...)
# ... repeat for every field
```

In FastAPI:

```python
def create_asset(asset: Asset):  # Pydantic validated `asset` before this body runs
    ...
```

The volume difference is obvious. The **shape** difference matters more:

- **Where the rule lives.** In Flask, the validation rule for `price` lives inside the view function that uses `price`. Add a second endpoint that also accepts a price and you copy/paste the rule (or extract a helper you have to remember to call). In FastAPI, the rule lives on the `Asset` model — every endpoint that accepts `Asset` gets the same validation for free.
- **What the client sees on failure.** Flask's `abort(422, description=...)` returns an HTML error page by default; you have to register an error handler to make it JSON. FastAPI returns a structured `{detail: [{loc, msg, type}, ...]}` for every Pydantic failure — automatically.
- **What the docs see.** The `Asset` model in FastAPI is the *same object* that drives validation, response shaping, and the OpenAPI schema (section 4). In Flask, the validation rule and the schema doc are two separate sources of truth, drifting from each other one PR at a time.

You can close the gap by adding a library — `marshmallow`, `pydantic` itself (Flask doesn't object to importing it), `flask-pydantic-spec`, or one of the API-extension toolkits. But that's adding back the integration FastAPI gives you out of the box.

In [2]:
# Same error condition; same payload; observe the shape of each framework's response.
import json as _json
bad = {"ticker": "TSLA", "name": "Tesla", "price": -1, "extra": "field"}

flask_resp = fc.post("/assets", json=bad)
print("FLASK 422 shape:")
print(f"  content-type : {flask_resp.headers.get('content-type')}")
print(f"  body (first 200 chars): {flask_resp.get_data(as_text=True)[:200]!r}")
print(f"  get_json()   : {flask_resp.get_json()!r}  # JSON parse returns None — this is HTML")

print("\nFASTAPI 422 shape:")
print(_json.dumps(ac.post("/assets", json=bad).json(), indent=2))

# And what about an *unknown* field? Flask's hand-rolled checker silently accepts
# the "extra" key. FastAPI accepts it too by default — but turning that off is one line.
print("\nFASTAPI 422 with `model_config = ConfigDict(extra=\"forbid\")`:")
from pydantic import ConfigDict
class StrictAsset(Asset):
    model_config = ConfigDict(extra="forbid")

strict_app = FastAPI()
@strict_app.post("/assets", status_code=201)
def strict_create(asset: StrictAsset):
    return asset

print(_json.dumps(TestClient(strict_app).post("/assets", json=bad).json(), indent=2))


FLASK 422 shape:
  content-type : text/html; charset=utf-8
  body (first 200 chars): '<!doctype html>\n<html lang=en>\n<title>422 Unprocessable Entity</title>\n<h1>Unprocessable Entity</h1>\n<p>price must be a non-negative number</p>\n'
  get_json()   : None  # JSON parse returns None — this is HTML

FASTAPI 422 shape:
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "price"
      ],
      "msg": "Input should be greater than or equal to 0",
      "input": -1,
      "ctx": {
        "ge": 0.0
      }
    }
  ]
}

FASTAPI 422 with `model_config = ConfigDict(extra="forbid")`:
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "price"
      ],
      "msg": "Input should be greater than or equal to 0",
      "input": -1,
      "ctx": {
        "ge": 0.0
      }
    },
    {
      "type": "extra_forbidden",
      "loc": [
        "body",
        "extra"
      ],
      "msg": "Extra inputs are not p

The takeaway: in FastAPI, "reject unknown fields" is a one-line `model_config = ConfigDict(extra="forbid")`. In Flask, you'd have to iterate `payload.keys()` against a hard-coded allowlist in every view — the kind of code that gets out of date the moment you add a field.

## 3. Async Support

Flask 3 supports `async def` views — but with caveats that make the support look more capable than it is.

What Flask does with an `async def` view: it runs the coroutine to completion **inside the WSGI worker thread**, by spinning up an event loop *per request* (via `asgiref`). The view is async; the server underneath is still synchronous, one request per worker thread.

What that means in practice:

- You can `await` inside a view — useful for calling an async client (`httpx.AsyncClient`) once or twice.
- You **do not get concurrency between requests for free.** Flask still allocates one worker thread per concurrent request. If the async work is mostly I/O wait, you've added overhead without buying parallelism.
- It is not the model the FastAPI/Uvicorn/Starlette stack uses. There, `async def` views are scheduled on a *shared* event loop and a single worker handles many concurrent I/O-bound requests interleaved — that's where the throughput win lives (notebook 3.1).

A concrete demo: an `async def` view in each framework that simulates a 100ms I/O wait. Fire 10 concurrent requests at each and look at the wall-clock difference.

In [3]:
import asyncio
import time
import httpx
from asgiref.wsgi import WsgiToAsgi

# ----- Flask async view -------------------------------------------------------
async_flask = Flask("async_flask_demo")

@async_flask.get("/slow")
async def f_slow():
    await asyncio.sleep(0.1)
    return jsonify({"ok": True})

# Wrap WSGI in ASGI so we can hit it with httpx.AsyncClient (a fair race).
flask_asgi = WsgiToAsgi(async_flask)

# ----- FastAPI async view -----------------------------------------------------
async_fa = FastAPI()

@async_fa.get("/slow")
async def fa_slow():
    await asyncio.sleep(0.1)
    return {"ok": True}

# ----- race -------------------------------------------------------------------
N = 10
async def race(transport_app, label):
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=transport_app),
                                  base_url="http://t") as client:
        t0 = time.perf_counter()
        await asyncio.gather(*(client.get("/slow") for _ in range(N)))
        dt = time.perf_counter() - t0
        print(f"{label:<22} {N} concurrent /slow -> {dt:.3f}s "
              f"({'serial-ish' if dt > 0.5 else 'concurrent'})")

await race(flask_asgi, "Flask  (async view)")
await race(async_fa,   "FastAPI (async view)")


Flask  (async view)    10 concurrent /slow -> 1.089s (serial-ish)
FastAPI (async view)   10 concurrent /slow -> 0.108s (concurrent)


On the FastAPI side, 10 × 100ms sleeps overlap on the single event loop — total around 0.1s plus overhead. On the Flask side the result is roughly 10 × 100ms = 1.0s: **the `async def` view ran, but request *concurrency* did not happen**. `WsgiToAsgi` is a 1:1 adapter — it lets an ASGI server *run* a WSGI app, but each request still walks through the synchronous WSGI pipeline serially. The `asyncio.sleep` inside the view returns control to *that view's* per-request event loop; the next request can't start until this one finishes.

The structural truth: in FastAPI, `async def` views share one event loop and the framework schedules them cooperatively. In Flask, `async def` views run in *isolated per-request event loops* sitting behind a synchronous request pipeline. The Flask shape is fine for "I need to `await` one HTTP call inside this view"; it is not the shape for "I need to fan out to 50 downstreams concurrently."

If async I/O concurrency is core to the service, FastAPI's event-loop model is the structural fit. If it's incidental, Flask 3's support is enough.


## 4. OpenAPI: Manual vs Free

This is the cleanest single comparison. FastAPI generates `/openapi.json` and serves `/docs` (Swagger UI) and `/redoc` automatically, derived from the type annotations on your routes. Flask generates *nothing* — you either write the schema by hand, or you add a library (`flask-smorest`, `flask-pydantic-spec`, `apispec`) and decorate every route.

The cell below pulls the OpenAPI schema from the FastAPI app and prints the asset endpoint description it inferred. Then we count the Flask endpoints that would need a hand-written description if you wanted a similar doc surface.

In [4]:
# FastAPI: free, complete, kept in sync with the code by virtue of *being* the code.
schema = ac.get("/openapi.json").json()
asset_post = schema["paths"]["/assets"]["post"]
print("FastAPI inferred POST /assets schema:")
print("  summary       :", asset_post.get("summary"))
print("  requestBody   :", _json.dumps(asset_post["requestBody"]["content"]["application/json"]["schema"]))
print("  responses     :", list(asset_post["responses"].keys()))

# Flask: nothing on disk. The routes know their URL rules but the framework
# has no idea what payloads they accept or return.
print("\nFlask: every registered URL rule (no schema attached):")
for rule in flask_app.url_map.iter_rules():
    if rule.endpoint == "static": continue
    methods = sorted(rule.methods - {"HEAD", "OPTIONS"})
    print(f"  {','.join(methods):<10} {rule.rule}   (no schema)")


FastAPI inferred POST /assets schema:
  summary       : Fa Create Asset
  requestBody   : {"$ref": "#/components/schemas/Asset"}
  responses     : ['201', '422']

Flask: every registered URL rule (no schema attached):
  GET        /assets   (no schema)
  GET        /assets/<ticker>   (no schema)
  POST       /assets   (no schema)


The Flask side prints the URL rules and nothing else — there is no machine-readable description of what `/assets` accepts or returns. The cost of that gap shows up in three places:

- **Client codegen.** OpenAPI is the input to `openapi-generator`, `oapi-codegen`, and friends. Flask without an extension can't produce a spec, so clients are written by hand and drift.
- **API gateway integration.** AWS API Gateway, Kong, Apigee, Tyk all consume OpenAPI to attach auth, rate-limit, and routing rules. Hand-written specs decay.
- **Contract tests.** Tools like `schemathesis` generate property-based tests from an OpenAPI spec; the more truthful your spec, the more bugs they find. A hand-maintained spec finds fewer because it's smaller and behind.

If you adopt `flask-smorest`, you get most of this back — at the price of decorating every route with explicit schema references, which is exactly what FastAPI's `def create_asset(asset: Asset)` is already doing implicitly.

## 5. When Flask Is Still Right

This is not a hit piece. Flask is the right choice for a non-trivial set of services:

- **You're already on Flask** and the migration would be more disruptive than the gains. The framework is mature, stable, and not going anywhere — Flask 3 came out in 2023 and `pallets-eco` still has active maintainers. A Flask app that works does not need to be rewritten.
- **You don't need an API at all — you need a small server-rendered web app.** Jinja templates, server-rendered HTML, sessions, CSRF, Flask-Login. FastAPI can do all of this, but Flask's ecosystem is genuinely deeper for the form-and-template shape; Flask-WTF, Flask-Admin, Flask-Login are decades of accumulated work.
- **Your team's collective experience is overwhelmingly Flask** and the next hire will be too. Tooling familiarity is a real productivity lever; "use the boring thing the team knows" is sometimes the right answer.
- **Embedded in a larger Pythonic stack** that already speaks WSGI: a Django app that grew an API, an Airflow plugin, a Sentry hook, a Plotly Dash dashboard. ASGI adapters exist but introduce a translation layer with its own edges (see section 3).
- **Sub-millisecond simplicity for a tiny webhook receiver.** A 20-line Flask app receiving one webhook and putting it on a queue is hard to beat for cognitive overhead. FastAPI's machinery would be deadweight.

The honest rule of thumb: if the service is API-first, expected to grow, externally consumed, or async-heavy — pick FastAPI. If it's template-heavy, internal, small, or already-Flask, leave it alone.

## 6. Migration Sketch

If you do decide to migrate, the strangler-fig pattern is the safest path: **mount the new FastAPI app under a path prefix in front of the existing Flask app**, move endpoints across one at a time, and the day the last Flask route is gone, drop the WSGI adapter.

The shape, with both apps in the same process:

In [5]:
# Migration scaffolding: FastAPI in front, Flask mounted under /legacy/* via WSGI middleware.
from fastapi.middleware.wsgi import WSGIMiddleware

# Pretend we've just migrated GET /assets to FastAPI but POST /assets is still on Flask.
migrate_app = FastAPI(title="Strangling Flask")

# New, migrated endpoint — already FastAPI-native:
@migrate_app.get("/assets", response_model=list[Asset])
def m_list_assets():
    return list(fa_store.values())

# Everything not yet ported falls through to the Flask app, mounted at /legacy.
# In production you'd mount Flask at "/" and put the FastAPI shim in front via
# nginx; the in-process variant is easier to demonstrate here.
legacy_flask = Flask("legacy_during_migration")
@legacy_flask.post("/assets")
def legacy_create():
    payload = request.get_json(silent=True) or {}
    # ... validation we haven't ported yet ...
    return jsonify({"created": payload, "via": "flask-legacy"}), 201

migrate_app.mount("/legacy", WSGIMiddleware(legacy_flask))

c = TestClient(migrate_app)
print("GET  /assets (FastAPI-native) :", c.get("/assets").json()[:1], "...")
print("POST /legacy/assets (Flask)   :", c.post("/legacy/assets", json={"ticker":"NVDA"}).json())


GET  /assets (FastAPI-native) : [{'ticker': 'AAPL', 'name': 'Apple Inc.', 'price': 190.0}] ...
POST /legacy/assets (Flask)   : {'created': {'ticker': 'NVDA'}, 'via': 'flask-legacy'}


That's the whole pattern. Three rules make it work without drama:

- **Move endpoints in domain-bounded batches.** All `/assets` together, all `/portfolios` together — never a half-migrated resource where half the verbs are on one framework and half on the other.
- **Pick the Pydantic schemas before you touch the routes.** The Pydantic model for `Asset` should be agreed-on before you migrate any endpoint that uses it; otherwise you'll find yourself rewriting the same model three times as you migrate three routes.
- **Tests come first.** Before you migrate a Flask route, write its FastAPI test (notebook 7.1). The migration is "passing tests stay passing." If you don't have tests on the Flask route, write Flask tests first (Werkzeug's `test_client()` is fine), confirm behaviour, then port.

The strangling can also go the *other* way at the URL level: keep Flask in front, put FastAPI under a `/v2/` prefix. Either direction works; the choice usually comes down to which framework owns the auth middleware you don't want to rewrite first.

## Key Takeaways

- **Same feature, different defaults.** Flask gives you primitives and asks you to assemble; FastAPI gives you assembled defaults and asks you to override. Both can produce identical APIs — the per-route cost is the differentiator.
- **Pydantic at the boundary replaces ~10 lines of manual validation per endpoint** with a 4-line model. The win compounds: the same model drives validation, response shaping, and the OpenAPI schema.
- **Flask 3 async views are real but limited.** `async def` works; concurrent I/O across requests does not come for free. If async I/O is core, FastAPI's event-loop model is the structural fit.
- **OpenAPI is the single biggest practical gap.** Free in FastAPI, only via extensions in Flask. The cost shows up in clients, gateways, and contract tests — not in Day 1 productivity.
- **Migration is endpoint-by-endpoint, not big-bang.** Strangler-fig with `WSGIMiddleware`: FastAPI in front, Flask mounted under a prefix, move endpoints in domain batches.
- **Flask remains the right call** for template-heavy apps, sub-millisecond webhook receivers, and stable existing Flask codebases where the migration would cost more than the gains.
- **Capstone tie-in**: the capstone is FastAPI-only by design — but the README will reference this notebook for teams arriving from Flask, and the migration recipe is the path they'd actually walk.

## Exercises

**1. Add `PATCH /assets/<ticker>` to both apps.** The Flask version takes the ticker from the path, reads JSON, manually validates that the payload only contains a subset of `{name, price}`, and applies a partial update. The FastAPI version uses a second Pydantic model `AssetPatch(BaseModel)` with optional fields and `model_dump(exclude_unset=True)`. Time the two using `timeit` over 1000 calls each and report the per-call delta. The point: PATCH is where Pydantic's "exclude unset" support pays for itself.

**2. Make Flask emit JSON error envelopes consistently.** Register a Flask error handler with `@flask_app.errorhandler(HTTPException)` that returns `jsonify({"detail": e.description}), e.code` for every abort, and confirm the 404, 422, and 415 responses now look uniform. The point: Flask can produce structured errors — it just doesn't by default.

**3. Generate OpenAPI for the Flask app via `flask-smorest`** (or `apispec`, your call) for the three endpoints in section 1. Compare the resulting schema to FastAPI's `/openapi.json` for the same endpoints — note any fields one has and the other doesn't (response examples, validation constraints, error responses). The point: parity is reachable on the Flask side, but you pay for it in per-route boilerplate.